# Detecting LLM-generated reviews in Lithuanian — experiment notebook

Run top-to-bottom in **Google Colab** (Runtime → Change runtime type → **T4 GPU**).

| Part | What it produces (for the report) | Approx. time on T4 |
|---|---|---|
| 1. Data audit | Tables 1–2 (composition, shortcut signals) | 2 min |
| 2. Splits & variants | Train/val/test + control variants | 1 min |
| 3. Classical models (TF-IDF, NB) | RQ1 rows | 5 min |
| 4. CNN & BiLSTM | RQ1 rows (3 seeds) | 20–30 min |
| 5. XLM-R | RQ1 row | 30–45 min |
| 6. Leave-one-generator-out | RQ2 table | 30–60 min |
| 7. Shortcut controls | RQ3 table | 30–60 min |
| 8. Export | CSVs, figures, error samples | 1 min |

Everything is written to `RESULTS_DIR` (mount Google Drive in cell 0 so you don't lose results if Colab disconnects).

In [ ]:
# 0. Setup ---------------------------------------------------------------
!pip -q install -U datasets transformers accelerate

import os, re, json, time, random, unicodedata
import numpy as np, pandas as pd

USE_DRIVE = True   # set False if you don't want Google Drive
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RESULTS_DIR = '/content/drive/MyDrive/lt_mgt_detection'
else:
    RESULTS_DIR = '/content/lt_mgt_detection'
os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    except ImportError:
        pass
set_seed(SEED)
print('Results will be saved to', RESULTS_DIR)

## 1. Load data and audit it (dataset has no documentation, so this is part of the contribution)

In [ ]:
from datasets import load_dataset
raw = load_dataset('engineerai/lithuanian-fake-review-detection', split='train').to_pandas()
print(raw.shape); print(raw.columns.tolist())
raw.head()

In [ ]:
# 1a. Basic composition -------------------------------------------------
df = raw.copy()
df['review_text'] = df['review_text'].fillna('').astype(str)
df['generator'] = np.where(df['source'].str.startswith('generated_'),
                           df['source'].str.replace('generated_', '', regex=False), 'human')

comp = pd.crosstab(df['source'], df['label'], margins=True)
print('Label x source:'); display(comp)
# Sanity check: label 1 should coincide exactly with generated sources
print('label/generator agreement:', ((df['label']==1) == (df['generator']!='human')).mean())
comp.to_csv(f'{RESULTS_DIR}/audit_label_by_source.csv')

In [ ]:
# 1b. Shortcut-signal audit ----------------------------------------------
LT_DIAC = set('ąčęėįšųūžĄČĘĖĮŠŲŪŽ')
CYR = re.compile(r'[\u0400-\u04FF]')

def feats(t):
    n = max(len(t), 1); w = t.split()
    return dict(
        n_chars=len(t), n_words=len(w),
        avg_word_len=np.mean([len(x) for x in w]) if w else 0,
        diac_ratio=sum(c in LT_DIAC for c in t)/n,
        punct_ratio=sum(unicodedata.category(c).startswith('P') for c in t)/n,
        digit_ratio=sum(c.isdigit() for c in t)/n,
        upper_ratio=sum(c.isupper() for c in t)/n,
        has_cyrillic=int(bool(CYR.search(t))),
        n_excl=t.count('!'), n_sent=len(re.findall(r'[.!?]+', t)) or 1,
    )

def add_feats(d):
    F = pd.DataFrame([feats(t) for t in d['review_text']], index=d.index)
    d[F.columns] = F
    return d, list(F.columns)

df, FEAT_COLS = add_feats(df.reset_index(drop=True))
df['store_missing'] = df['store'].isna().astype(int)

audit = df.groupby('generator')[FEAT_COLS+['store_missing']].mean().round(3)
display(audit)
audit.to_csv(f'{RESULTS_DIR}/audit_shortcut_signals.csv')

dups_all = df.duplicated('review_text').sum()
conflict = df.groupby('review_text')['label'].nunique().gt(1).sum()
print(f'exact duplicate texts: {dups_all}, texts appearing with BOTH labels: {conflict}')

**What to look at:** if, e.g., `has_cyrillic` or `store_missing` is >0 only for one class, or length distributions differ strongly,
the classifier can 'cheat'. Write these numbers into the Data chapter (Table 2) — this motivates RQ3.

In [ ]:
# 1c. Cleaning -------------------------------------------------------------
before = len(df)
conflict_texts = set(df.groupby('review_text')['label'].nunique().loc[lambda s: s>1].index)
df = df[~df['review_text'].isin(conflict_texts)]
df = df.drop_duplicates('review_text')
df = df[df['n_chars'] >= 3].reset_index(drop=True)
print(f'removed {before-len(df)} rows -> {len(df)} remain')
display(df.groupby(['generator']).size())

## 2. Splits and control variants

In [ ]:
from sklearn.model_selection import train_test_split
df['strat'] = df['generator']          # stratify by generator (keeps human/openai/anthropic/gemini proportions)
train_df, tmp = train_test_split(df, test_size=0.30, stratify=df['strat'], random_state=SEED)
val_df, test_df = train_test_split(tmp, test_size=0.50, stratify=tmp['strat'], random_state=SEED)
for n, d in [('train', train_df), ('val', val_df), ('test', test_df)]:
    print(n, len(d), d['label'].mean().round(3))

# --- Control variants for RQ3 (applied to train, val AND test) -----------
STRIP = str.maketrans('ąčęėįšųūžĄČĘĖĮŠŲŪŽ', 'aceeisuuzACEEISUUZ')
def normalize(t):
    t = t.lower().translate(STRIP)          # remove diacritics (real users often omit them)
    t = re.sub(r'\d+', '0', t)             # mask numbers / dates
    t = re.sub(r'[^\w\s]', ' ', t)        # drop punctuation
    return re.sub(r'\s+', ' ', t).strip()

def lt_only(d):
    return d[d['has_cyrillic'] == 0]

def length_matched(d, seed=SEED, bins=(0,5,10,20,40,80,10_000)):
    d = d.copy(); d['lbin'] = pd.cut(d['n_words'], bins=bins, right=False)
    parts = []
    for _, g in d.groupby('lbin', observed=True):
        k = g['label'].value_counts().min() if g['label'].nunique()==2 else 0
        if k: parts.append(g.groupby('label', group_keys=False).sample(k, random_state=seed))
    return pd.concat(parts).sample(frac=1, random_state=seed)

def make_variant(name):
    tr, va, te = train_df.copy(), val_df.copy(), test_df.copy()
    if name in ('lt_only', 'controlled'):
        tr, va, te = lt_only(tr), lt_only(va), lt_only(te)
    if name in ('length_matched', 'controlled'):
        tr, va, te = length_matched(tr), length_matched(va), length_matched(te)
    if name in ('normalized', 'controlled'):
        for d in (tr, va, te):
            d['review_text'] = d['review_text'].map(normalize)
            add_feats(d)                      # recompute stylometric features on the normalized text
    return tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True)

VARIANTS = ['original', 'lt_only', 'length_matched', 'normalized', 'controlled']
for v in VARIANTS:
    a, b, c = make_variant(v); print(f'{v:15s} train={len(a):6d} val={len(b):5d} test={len(c):5d}')

## Metrics: macro-F1, AUC ROC, TPR @ 5% FPR (threshold calibrated on validation), bootstrap 95% CI

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, roc_curve

def thr_at_fpr(y_val, p_val, fpr_target=0.05):
    fpr, tpr, thr = roc_curve(y_val, p_val)
    ok = np.where(fpr <= fpr_target)[0]
    return thr[ok[-1]] if len(ok) else 1.0

def evaluate(y, p, y_val, p_val, n_boot=1000, seed=SEED):
    y, p = np.asarray(y), np.asarray(p)
    t5 = thr_at_fpr(np.asarray(y_val), np.asarray(p_val))
    def m(yy, pp):
        return (f1_score(yy, (pp>=0.5).astype(int), average='macro'),
                roc_auc_score(yy, pp) if len(np.unique(yy))==2 else np.nan,
                ((pp>=t5) & (yy==1)).sum() / max((yy==1).sum(), 1))
    base = m(y, p)
    rng = np.random.default_rng(seed); boots = []
    for _ in range(n_boot):
        i = rng.integers(0, len(y), len(y)); boots.append(m(y[i], p[i]))
    lo, hi = np.nanpercentile(boots, 2.5, axis=0), np.nanpercentile(boots, 97.5, axis=0)
    f = lambda x: float(np.round(x, 4))
    return dict(macro_f1=f(base[0]), f1_lo=f(lo[0]), f1_hi=f(hi[0]),
                auc=f(base[1]), auc_lo=f(lo[1]), auc_hi=f(hi[1]),
                tpr_at_5fpr=f(base[2]), tpr_lo=f(lo[2]), tpr_hi=f(hi[2]))

RESULTS = []
def log(exp, model, variant, seed, metrics, **extra):
    row = dict(experiment=exp, model=model, variant=variant, seed=seed, **metrics, **extra)
    RESULTS.append(row)
    pd.DataFrame(RESULTS).to_csv(f'{RESULTS_DIR}/results.csv', index=False)
    print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in row.items()})

## 3. Classical baselines (TF-IDF + Logistic Regression, Naive Bayes, stylometric-only)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def classical_models():
    return {
        'tfidf_word_lr': make_pipeline(TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
                                       LogisticRegression(max_iter=2000, C=4)),
        'tfidf_char_lr': make_pipeline(TfidfVectorizer(analyzer='char_wb', ngram_range=(2,5), min_df=3, sublinear_tf=True),
                                       LogisticRegression(max_iter=2000, C=4)),
        'count_nb':      make_pipeline(CountVectorizer(ngram_range=(1,2), min_df=2), MultinomialNB()),
    }

STYLO = ['n_chars','n_words','avg_word_len','diac_ratio','punct_ratio','digit_ratio','upper_ratio','has_cyrillic','n_excl','n_sent']

def run_classical(tr, va, te, exp, variant):
    for name, mdl in classical_models().items():
        t0 = time.time(); mdl.fit(tr['review_text'], tr['label'])
        pv, pt = mdl.predict_proba(va['review_text'])[:,1], mdl.predict_proba(te['review_text'])[:,1]
        log(exp, name, variant, SEED, evaluate(te['label'], pt, va['label'], pv), train_sec=round(time.time()-t0,1))
    # stylometric-only probe: if this is high, the task is solvable WITHOUT reading the text content
    st = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    st.fit(tr[STYLO], tr['label'])
    log(exp, 'stylometric_lr', variant, SEED,
        evaluate(te['label'], st.predict_proba(te[STYLO])[:,1], va['label'], st.predict_proba(va[STYLO])[:,1]))

tr, va, te = make_variant('original')
run_classical(tr, va, te, 'RQ1', 'original')

## 4. CNN (Kim, 2014) and BiLSTM — trained from scratch, 3 seeds

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from collections import Counter
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'; print(DEV)

TOK = re.compile(r'\w+|[^\w\s]')
def build_vocab(texts, max_size=30000):
    c = Counter(w for t in texts for w in TOK.findall(t.lower()))
    itos = ['<pad>', '<unk>'] + [w for w, _ in c.most_common(max_size)]
    return {w: i for i, w in enumerate(itos)}

def encode(texts, vocab, max_len=100):
    X = np.zeros((len(texts), max_len), dtype=np.int64)
    for i, t in enumerate(texts):
        ids = [vocab.get(w, 1) for w in TOK.findall(t.lower())][:max_len]
        X[i, :len(ids)] = ids
    return torch.tensor(X)

class KimCNN(nn.Module):
    def __init__(self, V, E=200, F=100, ks=(3,4,5), p=0.5):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.convs = nn.ModuleList([nn.Conv1d(E, F, k) for k in ks])
        self.drop = nn.Dropout(p); self.fc = nn.Linear(F*len(ks), 1)
    def forward(self, x):
        e = self.emb(x).transpose(1, 2)
        h = [torch.relu(c(e)).max(dim=2).values for c in self.convs]
        return self.fc(self.drop(torch.cat(h, 1))).squeeze(1)

class BiLSTM(nn.Module):
    def __init__(self, V, E=200, H=128, p=0.3):
        super().__init__()
        self.emb = nn.Embedding(V, E, padding_idx=0)
        self.lstm = nn.LSTM(E, H, batch_first=True, bidirectional=True)
        self.drop = nn.Dropout(p); self.fc = nn.Linear(2*H, 1)
    def forward(self, x):
        h, _ = self.lstm(self.emb(x))
        mask = (x != 0).unsqueeze(2)
        h = h.masked_fill(~mask, -1e4).max(dim=1).values   # max-pool over real tokens
        return self.fc(self.drop(h)).squeeze(1)

@torch.no_grad()
def predict(model, X, bs=512):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(torch.sigmoid(model(X[i:i+bs].to(DEV))).cpu())
    return torch.cat(out).numpy()

def train_nn(arch, tr, va, epochs=5, bs=128, lr=1e-3, seed=SEED):
    set_seed(seed)
    vocab = build_vocab(tr['review_text'])
    Xtr, Xva = encode(tr['review_text'], vocab), encode(va['review_text'], vocab)
    ytr = torch.tensor(tr['label'].values, dtype=torch.float32)
    model = (KimCNN if arch=='cnn' else BiLSTM)(len(vocab)).to(DEV)
    opt = torch.optim.Adam(model.parameters(), lr=lr); lossf = nn.BCEWithLogitsLoss()
    dl = DataLoader(TensorDataset(Xtr, ytr), batch_size=bs, shuffle=True)
    best, best_state = -1, None
    for ep in range(epochs):
        model.train()
        for xb, yb in dl:
            opt.zero_grad(); loss = lossf(model(xb.to(DEV)), yb.to(DEV)); loss.backward(); opt.step()
        auc = roc_auc_score(va['label'], predict(model, Xva))
        print(f'  {arch} seed={seed} epoch={ep+1} val_auc={auc:.4f}')
        if auc > best: best, best_state = auc, {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, vocab

def run_nn(arch, tr, va, te, exp, variant, seeds=(13, 42, 2024)):
    for s in seeds:
        t0 = time.time(); model, vocab = train_nn(arch, tr, va, seed=s)
        pv = predict(model, encode(va['review_text'], vocab)); pt = predict(model, encode(te['review_text'], vocab))
        log(exp, arch, variant, s, evaluate(te['label'], pt, va['label'], pv), train_sec=round(time.time()-t0,1))

tr, va, te = make_variant('original')
for arch in ['cnn', 'bilstm']:
    run_nn(arch, tr, va, te, 'RQ1', 'original')

## 5. XLM-RoBERTa-base fine-tuning (plain PyTorch loop — works with any transformers version)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

XLMR = 'FacebookAI/xlm-roberta-base'
TRAIN_CAP = 30000      # subsample training set to keep a T4 run under ~40 min; set None for full data

def train_xlmr(tr, va, seed=SEED, epochs=2, bs=32, lr=2e-5, max_len=128):
    set_seed(seed)
    if TRAIN_CAP and len(tr) > TRAIN_CAP:
        tr = tr.groupby('generator', group_keys=False).sample(frac=TRAIN_CAP/len(tr), random_state=seed)
    tok = AutoTokenizer.from_pretrained(XLMR)
    model = AutoModelForSequenceClassification.from_pretrained(XLMR, num_labels=2).to(DEV)
    enc = tok(list(tr['review_text']), truncation=True, max_length=max_len, padding='max_length', return_tensors='pt')
    dl = DataLoader(TensorDataset(enc['input_ids'], enc['attention_mask'], torch.tensor(tr['label'].values)),
                    batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    sch = get_linear_schedule_with_warmup(opt, int(0.06*len(dl)*epochs), len(dl)*epochs)
    scaler = torch.amp.GradScaler('cuda', enabled=(DEV=='cuda'))
    for ep in range(epochs):
        model.train()
        for i, (ids, am, yb) in enumerate(dl):
            opt.zero_grad()
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEV=='cuda')):
                loss = model(input_ids=ids.to(DEV), attention_mask=am.to(DEV), labels=yb.to(DEV)).loss
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step()
            if i % 200 == 0: print(f'  epoch {ep+1} step {i}/{len(dl)} loss {loss.item():.4f}')
    return model, tok

@torch.no_grad()
def predict_xlmr(model, tok, texts, bs=128, max_len=128):
    model.eval(); out = []
    texts = list(texts)
    for i in range(0, len(texts), bs):
        e = tok(texts[i:i+bs], truncation=True, max_length=max_len, padding=True, return_tensors='pt').to(DEV)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(DEV=='cuda')):
            logits = model(**e).logits.float()
        out.append(torch.softmax(logits, -1)[:, 1].cpu())
    return torch.cat(out).numpy()

def run_xlmr(tr, va, te, exp, variant, seed=SEED):
    t0 = time.time(); model, tok = train_xlmr(tr, va, seed=seed)
    pv, pt = predict_xlmr(model, tok, va['review_text']), predict_xlmr(model, tok, te['review_text'])
    log(exp, 'xlmr_base', variant, seed, evaluate(te['label'], pt, va['label'], pv), train_sec=round(time.time()-t0,1))
    out = te[['review_text','label','generator']].copy(); out['p_machine'] = pt
    out.to_csv(f'{RESULTS_DIR}/preds_xlmr_{exp}_{variant}.csv', index=False)
    del model; torch.cuda.empty_cache()

tr, va, te = make_variant('original')
run_xlmr(tr, va, te, 'RQ1', 'original')

## 6. RQ2 — Leave-one-generator-out (train without generator *g*, test on human + *g*)

In [ ]:
GENS = [g for g in df['generator'].unique() if g != 'human']
RUN_XLMR_LOGO = True   # ~3 extra XLM-R runs; set False if short on GPU time

for g in GENS:
    tr = train_df[train_df['generator'] != g]
    va = val_df[val_df['generator'] != g]
    te = test_df[test_df['generator'].isin(['human', g])]
    print(f'--- held-out generator: {g}  (train={len(tr)}, test={len(te)})')
    m = classical_models()['tfidf_char_lr'].fit(tr['review_text'], tr['label'])
    log('RQ2', 'tfidf_char_lr', f'heldout_{g}', SEED,
        evaluate(te['label'], m.predict_proba(te['review_text'])[:,1], va['label'], m.predict_proba(va['review_text'])[:,1]))
    run_nn('cnn', tr, va, te, 'RQ2', f'heldout_{g}', seeds=(42,))
    if RUN_XLMR_LOGO:
        run_xlmr(tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True), 'RQ2', f'heldout_{g}')

## 7. RQ3 — Shortcut controls (how much performance survives when artefacts are removed?)

In [ ]:
RUN_XLMR_RQ3 = True
for v in ['lt_only', 'length_matched', 'normalized', 'controlled']:
    tr, va, te = make_variant(v)
    print(f'=== variant {v}: train={len(tr)} test={len(te)}')
    run_classical(tr, va, te, 'RQ3', v)
    run_nn('cnn', tr, va, te, 'RQ3', v, seeds=(42,))
    if RUN_XLMR_RQ3 and v == 'controlled':
        run_xlmr(tr, va, te, 'RQ3', v)

**Before Part 8:** if Colab disconnected mid-way, re-run cells 0–2 and the metrics cell, then
`RESULTS = pd.read_csv(f'{RESULTS_DIR}/results.csv').to_dict('records')` to restore earlier results.

## 8. Export tables, a figure and error samples for the report

In [ ]:
R = pd.DataFrame(RESULTS)
R.to_csv(f'{RESULTS_DIR}/results.csv', index=False)

agg = (R.groupby(['experiment','model','variant'])
         [['macro_f1','auc','tpr_at_5fpr']].agg(['mean','std']).round(4))
display(agg); agg.to_csv(f'{RESULTS_DIR}/results_aggregated.csv')

import matplotlib.pyplot as plt
rq1 = R[R.experiment=='RQ1'].groupby('model')['macro_f1'].agg(['mean','std']).sort_values('mean')
ax = rq1['mean'].plot.barh(xerr=rq1['std'].fillna(0), figsize=(6,3)); ax.set_xlabel('Macro-F1 (test)')
plt.tight_layout(); plt.savefig(f'{RESULTS_DIR}/fig_rq1_macro_f1.png', dpi=200); plt.show()

# Error analysis sample: 30 false positives (human flagged as AI) and 30 false negatives
p = pd.read_csv(f'{RESULTS_DIR}/preds_xlmr_RQ1_original.csv')
fp = p[(p.label==0) & (p.p_machine>=0.5)].sample(min(30, ((p.label==0)&(p.p_machine>=0.5)).sum()), random_state=SEED)
fn = p[(p.label==1) & (p.p_machine<0.5)].sample(min(30, ((p.label==1)&(p.p_machine<0.5)).sum()), random_state=SEED)
pd.concat([fp.assign(err='FP'), fn.assign(err='FN')]).to_csv(f'{RESULTS_DIR}/error_samples.csv', index=False)
print('Saved everything to', RESULTS_DIR)
print('Send back: results.csv, results_aggregated.csv, audit_*.csv, error_samples.csv, fig_rq1_macro_f1.png')